# Scraping: récupération de la page d'accueil de la RTBF

Dans ce notebook, nous créons un robot qui va ouvrir la page d'accueil du site du journal [La RTBF](https://www.rtbf.be/) et récupérer le titre de tous les articles du jour et les stocker dans un fichier csv.

## Imports

In [18]:
import os
import re
import time
import requests
from bs4 import BeautifulSoup
import pandas as pd

## Récupération de tous les articles de la page d'accueil



In [19]:
headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_10_1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/39.0.2171.95 Safari/537.36'}

articles = []

root_url = f"https://www.rtbf.be"
response = requests.get(root_url, headers=headers)
soup = BeautifulSoup(response.text, 'html.parser')
for link in soup.find_all("a", {"class":"stretched-link outline-hidden"}):
    title = link.text.strip()
    url = root_url + link.get('href')
    articles.append([url, title])

In [20]:
# Affichage du nombre d'articles récupérés
len(articles)

92

In [21]:
# Affichage des 10 premières entrées
articles[:10]

[['https://www.rtbf.behttps://www.rtbf.be/short/florent-brack-est-venu-nous-rendre-visite-sur-le-plateau-du-89-avec-son-titre-dawn-right-ca-vous-dit-quelque-chose-on-avoue-cest-la-musique-utilisee-pour-la-promo-du-89-et-de-cest-vous-qui-le-dites-6550?flux=main-1',
  ''],
 ['https://www.rtbf.behttps://www.rtbf.be/short/le-premier-defi-6546?flux=main-1',
  ''],
 ['https://www.rtbf.behttps://www.rtbf.be/short/les-enfants-de-choeur-6548?flux=main-1',
  ''],
 ['https://www.rtbf.behttps://www.rtbf.be/short/sorcieres-chronique-dun-massacre-6543?flux=main-1',
  ''],
 ['https://www.rtbf.behttps://www.rtbf.be/short/projet-pilote-de-40-bracelets-electroniques-pour-mineurs-6544?flux=main-1',
  ''],
 ['https://www.rtbf.behttps://www.rtbf.be/short/avoir-20-ans-en-1990-6542?flux=main-1',
  ''],
 ['https://www.rtbf.behttps://www.rtbf.be/short/le-tour-de-france-a-velo-dedwige-6545?flux=main-1',
  ''],
 ['https://www.rtbf.behttps://www.rtbf.be/short/completement-foot-pascal-ferre-auteur-de-la-face-cache

## Création d'un dataframe avec les liens et les titres des articles


In [22]:
df = pd.DataFrame(articles, columns=['link', 'title'])
df

,link,title
0,https://www.rtbf.behttps://www.rtbf.be/short/f...,
1,https://www.rtbf.behttps://www.rtbf.be/short/l...,
2,https://www.rtbf.behttps://www.rtbf.be/short/l...,
3,https://www.rtbf.behttps://www.rtbf.be/short/s...,
4,https://www.rtbf.behttps://www.rtbf.be/short/p...,
...,...,...
87,https://www.rtbf.behttps://auvio.rtbf.be/media...,Velvet : Le nouvel empire
88,https://www.rtbf.behttps://auvio.rtbf.be/media...,Objectif Monde
89,https://www.rtbf.behttps://auvio.rtbf.be/media...,Formule 1 - Grand Prix Etats-Unis
90,https://www.rtbf.behttps://auvio.rtbf.be/media...,Des trains pas comme les autres


In [23]:
# Sauvegarde du dataframe dans un fichier csv
df.to_csv(f"../data/rtbf_{time.strftime('%Y%m%d')}.csv", index=False)

OSError: Cannot save file into a non-existent directory: '..\data'

## Téléchargement d'un article et affichage du texte

In [24]:
# Récupération du premier lien du dataframe
article_url = df.iloc[0][0]

# Ouverture de la page
response = requests.get(article_url, headers=headers)
soup = BeautifulSoup(response.content, 'html.parser')
# Parsing du contenu
title = soup.find("h1").get_text(separator=' ')

paragraphs = []
for paragraph in soup.find_all("p", attrs={"class": None}):
    paragraphs.append(paragraph.get_text(separator=' ').strip())

content = "\n".join(paragraphs)
print(title)
print("==================================")
print(content)

KeyError: 0

### Nettoyage du texte à l'aide d'expressions régulières

In [25]:
# Suppression de tout ce qui se trouve entre parenthèses
clean_content = re.sub("\([^\)]+\)", " ", content)
# Suppression des espaces multiples
clean_content = re.sub("\s+", " ", clean_content)

print(content)
print("==================================")
print(clean_content)


<>:2: SyntaxWarning: invalid escape sequence '\('
<>:4: SyntaxWarning: invalid escape sequence '\s'
<>:2: SyntaxWarning: invalid escape sequence '\('
<>:4: SyntaxWarning: invalid escape sequence '\s'
C:\Users\asusz\AppData\Local\Temp\ipykernel_23680\3799268790.py:2: SyntaxWarning: invalid escape sequence '\('
  clean_content = re.sub("\([^\)]+\)", " ", content)
C:\Users\asusz\AppData\Local\Temp\ipykernel_23680\3799268790.py:4: SyntaxWarning: invalid escape sequence '\s'
  clean_content = re.sub("\s+", " ", clean_content)


NameError: name 'content' is not defined

### Création d'un fichier avec le contenu de l'article


In [27]:
with open("../data/rtbf_example.txt", "w") as writer:
    writer.write(f"{title}\n\n{clean_content}")

FileNotFoundError: [Errno 2] No such file or directory: '../data/rtbf_example.txt'

## Pour en savoir plus

- Le web scraping avec Python: https://realpython.com/beautiful-soup-web-scraper-python/
- Tutoriel sur les expressions régulières: https://www.w3schools.com/python/python_regex.asp